# 17 — Semi-Autoregressive Drafting

**Engineering statement:** **Semi-autoregressive drafting specifies schedulable prefixes.**

Notebook 00 introduced the decoding problem. Notebook 07 reframed verification as a scarce engineering resource. Notebook 13 showed how confidence schedules that resource. Notebook 17 asks where the schedulable confidence profile comes from.

The core claim is:

> **Draft architecture specifies the confidence profile available to the verification scheduler.**

A drafter is not only a fast proposal module. In confidence-scheduled decoding, the drafter creates the operating regime that the scheduler can exploit.


## Repository roadmap

```text
00 Context
        ↓
07 Verification Resource
        ↓
13 Confidence Scheduling
        ↓
17 Semi-Autoregressive Drafting
        ↓
23 Throughput Objective
        ↓
29 Hardware-Aware Scheduling
```


## Start here

```text
parallel backbone
        ↓
local sequential head
        ↓
conditional confidence
        ↓
prefix survival
        ↓
verification scheduler
        ↓
accepted tokens
        ↓
higher throughput
```

Notebook 13 treated confidence scores as inputs to a scheduler. Notebook 17 studies how the draft model shapes those inputs.


In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch

# Robust repo paths: works from repo root, notebooks/, or uploaded standalone notebook.
CWD = Path.cwd().resolve()
if (CWD / "notebooks").exists() and (CWD / "figures").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "notebooks").exists() and (CWD.parent / "figures").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "17_semi_autoregressive_drafting"
SITE = REPO_ROOT / "site" / "2606-19348"
SITE_IMAGES = SITE / "images"

for path in [NOTEBOOKS, FIGURES, RESULTS, SITE_IMAGES]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 1. Why semi-autoregressive drafting?

A speculative decoder has three competing pressures:

| Drafting strategy | Strength | Weakness |
|---|---|---|
| Autoregressive draft | strong local dependence | sequential draft cost |
| Parallel draft | fast block proposal | weak intra-block conditioning |
| Semi-autoregressive draft | fast backbone plus local refinement | extra lightweight draft computation |

The design goal is not simply to draft more tokens. The goal is to draft tokens that remain useful under confidence-scheduled verification.


In [ ]:
def savefig(fig, name):
    path = FIGURES / name
    fig.savefig(path, dpi=180, bbox_inches="tight")
    try:
        site_path = SITE_IMAGES / name
        fig.savefig(site_path, dpi=180, bbox_inches="tight")
    except Exception:
        pass
    plt.show()
    print(path)
    return path

fig, ax = plt.subplots(figsize=(12, 5.8))
ax.axis("off")
ax.set_title("Why semi-autoregressive drafting?", fontsize=22, pad=18)

rows = [
    ("Autoregressive draft", "token → token → token → token", "strong local dependence\nsequential draft cost"),
    ("Parallel drafter", "[ token token token token ]", "single draft pass\nweak intra-block conditioning"),
    ("Semi-autoregressive", "parallel block + light local head", "fast draft backbone\nrecover local transitions"),
]
y_positions = [0.72, 0.48, 0.24]

for y, (label, center, note) in zip(y_positions, rows):
    ax.text(0.03, y, label, ha="left", va="center", fontsize=16, fontweight="bold")
    ax.text(
        0.39, y, center,
        ha="center", va="center", fontsize=16,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.2),
    )
    ax.text(0.72, y, note, ha="left", va="center", fontsize=15)

ax.text(
    0.5, 0.06,
    "Design goal: keep drafting parallel while restoring enough prefix dependence for scheduling.",
    ha="center", va="center", fontsize=15,
)

fig.tight_layout()
block_generation_fig = savefig(fig, "17_block_generation.png")


The semi-autoregressive design keeps the heavy computation parallel while adding enough local structure for the scheduler to trust longer prefixes.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.4))
ax.axis("off")
ax.set_title("Semi-autoregressive architecture flow", fontsize=22, pad=20)

steps = [
    "Target\ncontext",
    "Parallel\nbackbone",
    "Local\nsequential head",
    "Conditional\nconfidence",
    "Schedulable\nprefix",
]
xs = np.linspace(0.08, 0.92, len(steps))
y = 0.55
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(
        x, y, label,
        ha="center", va="center", fontsize=15,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.3),
    )
    if i < len(steps) - 1:
        ax.annotate("", xy=(xs[i+1]-0.09, y), xytext=(x+0.09, y), arrowprops=dict(arrowstyle="->", lw=2))

ax.text(
    0.5, 0.13,
    "Parallel computation produces candidate tokens; a lightweight sequential head restores local conditional structure before scheduling.",
    ha="center", va="center", fontsize=15,
)
fig.tight_layout()
architecture_flow_fig = savefig(fig, "17_architecture_flow.png")


## 2. Draft architecture shapes conditional confidence

A verification scheduler does not see the internal architecture directly. It sees a confidence profile.

Conditional confidence for token position \(j\) is written as

\[
c_j = P(y_j \mid y_{<j}, x).
\]

A parallel drafter can propose many tokens quickly, but confidence often decays across the suffix. A fully autoregressive draft can maintain strong conditional structure, but costs more sequential work. A semi-autoregressive draft is valuable when it preserves enough suffix confidence without paying the full autoregressive cost.


In [ ]:
positions = np.arange(1, 9)
profiles = pd.DataFrame({
    "position": positions,
    "parallel": [0.94, 0.82, 0.66, 0.46, 0.27, 0.13, 0.05, 0.02],
    "autoregressive": [0.86, 0.86, 0.85, 0.85, 0.84, 0.84, 0.83, 0.83],
    "semi_autoregressive": [0.94, 0.91, 0.88, 0.84, 0.79, 0.72, 0.65, 0.58],
})
profiles_path = RESULTS / "17_confidence_profiles.csv"
profiles.to_csv(profiles_path, index=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
for col in ["parallel", "autoregressive", "semi_autoregressive"]:
    ax.plot(profiles["position"], profiles[col], marker="o", linewidth=2, label=col)
ax.set_title("Draft architecture shapes conditional confidence", fontsize=18)
ax.set_xlabel("Draft position")
ax.set_ylabel(r"Conditional confidence $c_j$")
ax.set_ylim(0.0, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
confidence_profiles_fig = savefig(fig, "17_confidence_profiles.png")
profiles


The scheduler does not choose from raw token probabilities independently. It needs prefix-level survival.

A prefix only survives if all earlier draft tokens survive. Conditional confidence therefore compounds:

\[
a_j = \prod_{k=1}^{j} c_k.
\]


In [ ]:
survival = profiles.copy()
for col in ["parallel", "autoregressive", "semi_autoregressive"]:
    survival[col] = np.cumprod(profiles[col].to_numpy())

survival_path = RESULTS / "17_prefix_survival.csv"
survival.to_csv(survival_path, index=False)
threshold = 0.35

fig, ax = plt.subplots(figsize=(10, 5.5))
for col in ["parallel", "autoregressive", "semi_autoregressive"]:
    ax.plot(survival["position"], survival[col], marker="o", linewidth=2, label=col)
ax.axhline(threshold, linestyle="--", linewidth=1.5, label="example scheduling floor")
ax.set_title("Conditional confidence compounds into prefix survival", fontsize=18)
ax.set_xlabel("Draft position")
ax.set_ylabel(r"Prefix survival $a_j$")
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
prefix_survival_fig = savefig(fig, "17_prefix_survival.png")
survival


## 3. Prefix survival becomes schedulable length

A schedulable prefix length can be defined by a floor on prefix survival:

\[
\ell(q)=\max\{j: a_j \ge q\}.
\]

The same draft block can therefore support different scheduling decisions depending on the system's risk tolerance and load.


In [ ]:
def sched_len(values, floor):
    ok = np.where(np.asarray(values) >= floor)[0]
    return int(ok[-1] + 1) if len(ok) else 0

floors = [0.50, 0.35, 0.20]
rows = []
for floor in floors:
    for col in ["parallel", "autoregressive", "semi_autoregressive"]:
        rows.append({"floor": floor, "strategy": col, "scheduled_prefix_length": sched_len(survival[col], floor)})
prefix_gain_df = pd.DataFrame(rows)
prefix_gain_path = RESULTS / "17_prefix_gain.csv"
prefix_gain_df.to_csv(prefix_gain_path, index=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
width = 0.24
x = np.arange(len(floors))
for i, col in enumerate(["parallel", "autoregressive", "semi_autoregressive"]):
    vals = prefix_gain_df[prefix_gain_df["strategy"] == col]["scheduled_prefix_length"].to_numpy()
    ax.bar(x + (i-1)*width, vals, width=width, label=col)
ax.set_xticks(x)
ax.set_xticklabels([fr"$a_j \geq {floor:.2f}$" for floor in floors])
ax.set_ylabel("Scheduled prefix length")
ax.set_title("Better draft profiles produce longer schedulable prefixes", fontsize=18)
ax.legend()
fig.tight_layout()
prefix_gain_fig = savefig(fig, "17_prefix_gain.png")
prefix_gain_df


The result is not simply "longer drafts." It is longer **useful** drafts: candidate prefixes that justify target-model verification.


## 4. Draft quality and draft cost jointly shape throughput

A better confidence profile only helps if the draft remains cheap enough. Notebook 17 therefore introduces the bridge objective for Notebook 23:

\[
\Theta(\ell)=
\frac{
\sum_{j=1}^{\ell} a_j
}{
T_{\rm draft}(\ell)+T_{\rm verify}(\ell)
}.
\]

The numerator is expected accepted draft tokens. The denominator is total drafting plus verification cost.


In [ ]:
def throughput_curve(a, draft_base, draft_slope, verify_cost):
    vals = []
    for ell in range(1, len(a)+1):
        accepted = np.sum(a[:ell])
        cost = draft_base + draft_slope * ell + verify_cost * ell
        vals.append(accepted / cost)
    return np.array(vals)

costs = {
    "parallel": (0.32, 0.015, 0.17),
    "autoregressive": (0.42, 0.11, 0.17),
    "semi_autoregressive": (0.36, 0.045, 0.17),
}
throughput_rows = []
fig, ax = plt.subplots(figsize=(10, 5.5))
for col in ["parallel", "autoregressive", "semi_autoregressive"]:
    vals = throughput_curve(survival[col].to_numpy(), *costs[col])
    ax.plot(positions, vals, marker="o", linewidth=2, label=col)
    best = int(np.argmax(vals) + 1)
    throughput_rows.append({"strategy": col, "best_prefix_length": best, "best_throughput_proxy": float(vals.max())})
ax.set_title("Draft quality and draft cost jointly shape throughput", fontsize=18)
ax.set_xlabel(r"Scheduled prefix length $\ell$")
ax.set_ylabel("Throughput proxy")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
drafting_vs_throughput_fig = savefig(fig, "17_drafting_vs_throughput.png")
throughput_df = pd.DataFrame(throughput_rows)
throughput_path = RESULTS / "17_drafting_vs_throughput.csv"
throughput_df.to_csv(throughput_path, index=False)
throughput_df


The semi-autoregressive curve is useful because it occupies the middle engineering regime:

- more schedulable suffix structure than fully parallel drafting,
- lower sequential cost than fully autoregressive drafting,
- a confidence profile that gives the scheduler more usable operating points.


## 5. Draft generation creates the scheduler input

Notebook 13 optimized a schedule. Notebook 17 specifies the object being scheduled.

```text
Prompt
  ↓
Draft model
  ↓
Candidate block
  ↓
Confidence profile
  ↓
Scheduled prefix
  ↓
Target verification
```


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.axis("off")
ax.set_title("Draft generation creates the scheduler input", fontsize=22, pad=20)
steps = ["Prompt", "Draft\nmodel", "Candidate\nblock", "Confidence\nprofile", "Scheduled\nprefix", "Target\nverification"]
xs = np.linspace(0.07, 0.93, len(steps))
y = 0.55
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, y, label, ha="center", va="center", fontsize=15,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.3))
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.085, y), xytext=(x+0.085, y), arrowprops=dict(arrowstyle="->", lw=2))
ax.text(0.5, 0.12,
        "The scheduler operates on confidence-structured draft blocks, not tokens alone.",
        ha="center", va="center", fontsize=15)
fig.tight_layout()
generation_pipeline_fig = savefig(fig, "17_generation_pipeline.png")


The architecture can be read as a causal chain:

\[
\text{architecture}
\rightarrow
\text{conditional confidence}
\rightarrow
\text{prefix survival}
\rightarrow
\text{verification schedule}
\rightarrow
\text{throughput}.
\]


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.0))
ax.axis("off")
ax.set_title("Confidence-scheduled decoding stack", fontsize=22, pad=18)
steps = [
    "Prompt / context",
    "Draft model",
    "Semi-autoregressive refinement",
    "Confidence head",
    "Verification scheduler",
    "Target model verification",
    "Accepted tokens",
]
ys = np.linspace(0.83, 0.18, len(steps))
x = 0.5
for i, (y, label) in enumerate(zip(ys, steps)):
    ax.text(x, y, label, ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.38", fc="white", ec="black", lw=1.3))
    if i < len(steps)-1:
        ax.annotate("", xy=(x, ys[i+1]+0.045), xytext=(x, y-0.045), arrowprops=dict(arrowstyle="->", lw=2))
ax.text(0.5, 0.06, "Drafting specifies the operating regime available to the scheduler.",
        ha="center", va="center", fontsize=15)
fig.tight_layout()
decoding_stack_fig = savefig(fig, "17_decoding_stack.png")


## Key equations

Conditional confidence:

\[
c_j=P(y_j\mid y_{<j},x)
\]

Prefix survival:

\[
a_j=\prod_{k=1}^{j}c_k
\]

Expected accepted prefix:

\[
A(\ell)=\sum_{j=1}^{\ell}a_j
\]

Engineering throughput objective:

\[
\Theta(\ell)=
\frac{A(\ell)}
{T_{\rm draft}(\ell)+T_{\rm verify}(\ell)}
\]

Notebook 23 will make this objective explicit as an operating-regime problem.


## Engineering summary

Semi-autoregressive drafting matters because it changes what confidence scheduling can do.

| Layer | Engineering role |
|---|---|
| Parallel backbone | keeps proposal generation fast |
| Local sequential head | restores intra-block conditional structure |
| Confidence head | exposes schedulable uncertainty |
| Verification scheduler | allocates target-model computation |
| Target verification | converts schedulable prefixes into accepted tokens |

The notebook's central statement is therefore:

> **Draft architecture specifies the confidence profile available to the verification scheduler.**


In [ ]:
summary = pd.DataFrame([
    {"notebook": "00_context", "question": "Why confidence-scheduled decoding?", "next": "verification as resource"},
    {"notebook": "07_verification_resource", "question": "What resource is scarce?", "next": "confidence scheduling"},
    {"notebook": "13_confidence_scheduling", "question": "How should verification be allocated?", "next": "draft architecture"},
    {"notebook": "17_semi_autoregressive_drafting", "question": "How should drafts be generated for scheduling?", "next": "throughput objective"},
    {"notebook": "23_throughput_objective", "question": "What objective should the scheduler optimize?", "next": "hardware-aware scheduling"},
])
summary_path = RESULTS / "17_repository_roadmap.csv"
summary.to_csv(summary_path, index=False)
summary


## Save notebook manifest

This manifest records the artifacts produced by Notebook 17.


In [ ]:
figures = {
    "block_generation": str(block_generation_fig.relative_to(REPO_ROOT)),
    "architecture_flow": str(architecture_flow_fig.relative_to(REPO_ROOT)),
    "confidence_profiles": str(confidence_profiles_fig.relative_to(REPO_ROOT)),
    "prefix_survival": str(prefix_survival_fig.relative_to(REPO_ROOT)),
    "prefix_gain": str(prefix_gain_fig.relative_to(REPO_ROOT)),
    "drafting_vs_throughput": str(drafting_vs_throughput_fig.relative_to(REPO_ROOT)),
    "generation_pipeline": str(generation_pipeline_fig.relative_to(REPO_ROOT)),
    "decoding_stack": str(decoding_stack_fig.relative_to(REPO_ROOT)),
}
manifest = {
    "notebook": "17_semi_autoregressive_drafting_v3.ipynb",
    "title": "Semi-Autoregressive Drafting",
    "engineering_statement": "Semi-autoregressive drafting specifies schedulable prefixes.",
    "central_claim": "Draft architecture specifies the confidence profile available to the verification scheduler.",
    "figures": figures,
    "tables": {
        "confidence_profiles": str(profiles_path.relative_to(REPO_ROOT)),
        "prefix_survival": str(survival_path.relative_to(REPO_ROOT)),
        "prefix_gain": str(prefix_gain_path.relative_to(REPO_ROOT)),
        "throughput": str(throughput_path.relative_to(REPO_ROOT)),
        "roadmap": str(summary_path.relative_to(REPO_ROOT)),
    },
    "next_notebook": "23_throughput_objective.ipynb",
}
manifest_path = RESULTS / "17_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## Download artifacts

Run the final cell to package Notebook 17 outputs for download.


In [ ]:
from IPython.display import FileLink, display

zip_path = RESULTS / "17_semi_autoregressive_drafting_artifacts.zip"
notebook_path = NOTEBOOKS / "17_semi_autoregressive_drafting_v3.ipynb"
fallback_notebook_path = Path.cwd() / "17_semi_autoregressive_drafting_v3.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    RESULTS,
]
for fig_path in figures.values():
    paths_to_include.append(REPO_ROOT / fig_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        arcname = path.relative_to(REPO_ROOT)
                    except ValueError:
                        arcname = path.name
                    zf.write(path, arcname)
        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                arcname = item.relative_to(REPO_ROOT)
            except ValueError:
                arcname = item.name
            zf.write(item, arcname)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
